# 差分メモ

- 対象: `[2]T5-v1.ipynb`
- 元: `[1]dpc-byt5-retrain-tokenizer-v3.ipynb`
- 種別: 枝分かれ
- 変更点:
  - 学習データを curated 版 `train.curated.v002-6.xlsx` のみに差し替え（`oare_id`, `transliteration`, `translation` を使用）
  - 前処理/後処理を `[2-7-2]dpc-starter-train-v3.ipynb` と同じ実装へ置き換え
  - SentencePiece 学習用コーパス作成・学習時前処理・推論時後処理を上記に統一
  - `MAX_LENGTH` を学習時/推論時とも `1024` に変更
- 変更理由/仮説: 長文分割反映データと統一前処理を使い、Tokenizer/T5 スクラッチ学習でも長文を欠落させにくくする


# Deep Past Initiative – Machine Translation

## Takeaway

- When the first letter of a word is capitalized it implies the word is a personal name or a place name (i.e. proper noun). When the word is in ALL CAPS, that implies it is a Sumerian logogram and was written in place of the Akkadian syllabic spelling for scribal simplicity.

- Can use the published_text's transliteration as the cleaned version for training.

- Can use the normed words in Lexicon to swap for training.

## Facts

- The transliteration words len < 150, translation words len < 300


In [ ]:
!pip install -q evaluate sacrebleu

In [ ]:
import os 
import gc
import re
import math
import unicodedata
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from sentence_transformers import SentenceTransformer, util
import evaluate
import matplotlib.pyplot as plt
import sentencepiece as spm
from transformers import T5Tokenizer

In [ ]:
class Config:
    # SentencePieceで学習したトークナイザ + T5をスクラッチ学習する実験
    # - 事前学習済みの ByT5 重みは使わない（このノートブックでは tokenizer も作り直す）
    # - `MODEL_NAME` は「この実験の識別子」としてのみ使う

    # ===== Model size preset =====
    # P100(16GB)で回しやすい順（目安）:
    # - p100-medium（推奨）: d_model=512, layers=12
    # - t5-base-p100        : d_model=768, layers=12（遅い/OOMしやすい）
    MODEL_PRESET = "p100-medium"

    MODEL_NAME = f"t5-scratch-spm-{MODEL_PRESET}"

    # 長文分割済みデータを活かすため、学習時/推論時とも 1024 に揃える
    MAX_LENGTH = 1024

    # ===== Training (P100想定) =====
    # 大型化に伴い、per-device batch を小さくして grad accum で稼ぐ
    BATCH_SIZE = 1
    GRAD_ACCUMULATION_STEPS = 8

    EPOCHS = 100
    LEARNING_RATE = 2e-4

    OUTPUT_DIR = f"./{MODEL_NAME}-akkadian-model"

    PATIENCE = 10

    DRY_RUN = False


In [ ]:
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything()

In [ ]:
INPUT_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"

# ------------------------------------------
# Train: curated xlsx を優先して読む
# ------------------------------------------
# Kaggle 上で使う場合は、curated xlsx を Dataset として追加して、
# `/kaggle/input/.../train.curated.v002-6.xlsx` が参照できるようにしてください。
USE_CURATED_TRAIN = True
CURATED_TRAIN_XLSX_CANDIDATES = [
    "/kaggle/input/datasets/tatsuokoshida/dpc-curated-train/train.curated.v002-6.xlsx",
]


def load_base_train_df():
    if USE_CURATED_TRAIN:
        curated_path = None
        for p in CURATED_TRAIN_XLSX_CANDIDATES:
            if os.path.exists(p):
                curated_path = p
                break
        if curated_path is None:
            raise FileNotFoundError(
                "curated train xlsx が見つかりません。CURATED_TRAIN_XLSX_CANDIDATES を確認してください。"
            )

        df = pd.read_excel(curated_path)

        unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed")]
        if unnamed_cols:
            df = df.drop(columns=unnamed_cols)

        df = df.rename(columns={"transliteration_x": "transliteration"})

        required_cols = {"oare_id", "transliteration", "translation"}
        missing_cols = required_cols - set(df.columns)
        assert not missing_cols, f"curated xlsx の必須カラムが不足しています: {sorted(missing_cols)}"

        df = df[["oare_id", "transliteration", "translation"]].copy()
        df["transliteration"] = df["transliteration"].fillna("").astype(str)
        df["translation"] = df["translation"].fillna("").astype(str)

        df.attrs["train_source"] = curated_path
        return df

    df = pd.read_csv(f"{INPUT_DIR}/train.csv")
    df.attrs["train_source"] = f"{INPUT_DIR}/train.csv"
    return df


base_train_df = load_base_train_df()
test_df = pd.read_csv(f"{INPUT_DIR}/test.csv")

train_df = base_train_df.copy()

train_source = base_train_df.attrs.get("train_source", "(unknown)")
print(f"Train Data: {len(train_df)} docs. | source={train_source}")

# Build Corpus

In [ ]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
from typing import List

SRC_COL = "transliteration"
TGT_COL = "translation"

PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

# --- Pre/Post processing (aligned to notebooks/006/lb-35-9-ensembling-post-processing-baseline.ipynb) ---
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


# Unicode fraction map (all single-character Unicode vulgar fractions)
_UNICODE_FRAC_MAP = {
    (1, 2): "½", (1, 3): "⅓", (2, 3): "⅔",
    (1, 4): "¼", (3, 4): "¾",
    (1, 5): "⅕", (2, 5): "⅖", (3, 5): "⅗", (4, 5): "⅘",
    (1, 6): "⅙", (5, 6): "⅚",
    (1, 7): "⅐",
    (1, 8): "⅛", (3, 8): "⅜", (5, 8): "⅝", (7, 8): "⅞",
    (1, 9): "⅑",
    (1, 10): "⅒",
}
def _trunc_float(x: float, places: int = 4) -> float:
    factor = 10 ** places
    if x >= 0:
        return math.floor(x * factor) / factor
    return -math.floor(-x * factor) / factor

def _fmt_trunc(x: float, places: int = 4) -> str:
    return f"{_trunc_float(x, places):.{places}f}".rstrip("0").rstrip(".")

# Map the 4-decimal *truncated* representation to a Unicode fraction.
# (Host example: 1.333330000... -> 1.3333; 0.1666 -> ⅙)
_FRAC_DECIMAL_MAP_4 = {}
for (n, d), ch in _UNICODE_FRAC_MAP.items():
    _FRAC_DECIMAL_MAP_4[_fmt_trunc(n / d, 4)] = ch

# Decimals (incl. long floats)
_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d+)(?![\w/])")
_LONG_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d{5,})(?![\w/])")

# General fractions like "1/2" or mixed fractions like "2 1/2".
_MIXED_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s+(\d+)\s*/\s*(\d+)(?![\w/])")
_SIMPLE_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s*/\s*(\d+)(?![\w/])")

def _mixed_frac_repl(m: re.Match) -> str:
    ip = int(m.group(1))
    num = int(m.group(2))
    den = int(m.group(3))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return f"{ip} {ch}" if ip != 0 else ch
    return m.group(0)

def _simple_frac_repl(m: re.Match) -> str:
    num = int(m.group(1))
    den = int(m.group(2))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return ch
    return m.group(0)

def _canon_decimal_str(s: str) -> str:
    # Keep x.5 (e.g., 2.5) as-is by request.
    if re.fullmatch(r"[1-9]\d*\.5", s):
        return s

    m = re.fullmatch(r"(\d+)\.(\d+)", s)
    if not m:
        return s

    ip = int(m.group(1))
    frac_digits = m.group(2)

    # Truncate fractional part to 4 digits (no rounding), pad with zeros for lookup.
    frac4 = (frac_digits + "0000")[:4]
    frac_key = ("0." + frac4).rstrip("0").rstrip(".")
    ch = _FRAC_DECIMAL_MAP_4.get(frac_key)
    if ch and frac_key != "0":
        if ip == 0:
            return ch
        return f"{ip} {ch}"

    # Long-float shortening: truncate to 4 digits after the decimal.
    if len(frac_digits) > 4:
        return f"{ip}.{frac4}".rstrip("0").rstrip(".")

    return s


_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_TAG_GAP_RE    = re.compile(r"<\s*gap\s*>", re.I)
_BARE_BIGGAP   = re.compile(r"\bbig[\s_\-]*gap\b", re.I)
_ELLIPSIS_RE   = re.compile(r"(?:\.{3,}|…+|\[\.+\])")
_BRACKET_X_RE  = re.compile(r"(\[\s*x\s*\]|\(\s*x\s*\))", re.I)
_XTOKEN_RUN_RE = re.compile(r"\bx(?:\s+x)+\b", re.I)
_XRUN_RE       = re.compile(r"(?<!\w)x{2,}(?!\w)", re.I)
_XTOK_RE       = re.compile(r"(?<!\w)x(?!\w)", re.I)
_WS_RE         = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)


_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")


class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)

        # Uppercase determinatives are unwrapped, lowercase ones are converted to braces.
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)

        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        ser = ser.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        ser = ser.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_QUOTES_RE = re.compile("[\u201c\u201d\u2018\u2019]")

_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}

_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")

_FORBIDDEN_TRANS = str.maketrans("", "", '()——<>⌈⌋⌊[]+ʾ;')

_COMMODITY_RE = re.compile(r'-(gold|tax|textiles)\b')
_COMMODITY_REPL = {
    "gold": "pašallum gold",
    "tax": "šadduātum tax",
    "textiles": "kutānum textiles",
}

def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]


_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅔ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

_ALT_SLASH_PAIR_RE = re.compile(r"\b([^\W\d_]+)\s*/\s*([^\W\d_]+)\b")
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)

        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)

        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)

        s = s.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        s = s.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        s = s.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)

        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)

        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        # Keep the left alternative (e.g., "you / she" -> "you")
        s = s.str.replace(_ALT_SLASH_PAIR_RE, r"\1", regex=True)

        # Remove only curly quotes; keep straight quotes and apostrophes.
        s = s.str.replace(_CURLY_QUOTES_RE, "", regex=True)

        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)

        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)

        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)

        s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()

        return s.tolist()

_preprocessor = OptimizedPreprocessor()
_postprocessor = VectorizedPostprocessor()




def clean_transliteration(s: str) -> str:
    return _preprocessor.preprocess_batch([s])[0]


def clean_translation(s: str) -> str:
    return _postprocessor.postprocess_batch([s])[0]


def postprocess_translation(text: str) -> str:
    return clean_translation(text)


with open("spm_corpus.txt", "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        f.write(clean_transliteration(row[SRC_COL]) + "\n")
        f.write(clean_translation(row[TGT_COL]) + "\n")


# Sentencepiece Tokenizer

In [ ]:
spm.SentencePieceTrainer.train(
    input="/kaggle/working/spm_corpus.txt",
    model_prefix="akkadian_spm",
    vocab_size=3000,
    model_type="unigram",         # better for low-resource
    character_coverage=1.0,
    user_defined_symbols=[
        "<DIV>", "<NUM>"
    ],
    bos_id=-1,
    eos_id=1,
    pad_id=0,
    unk_id=2
)

# Load Tokenizer

In [ ]:
tokenizer = T5Tokenizer(
    vocab_file="/kaggle/working/akkadian_spm.model",
    extra_ids=0,
    pad_token="<pad>",
    eos_token="</s>",
    unk_token="<unk>"
)

# T5 model for training

In [ ]:
from transformers import T5Config, T5ForConditionalGeneration

MODEL_PRESETS = {
    # 元ノート（小型）
    "original-small": {
        "d_model": 256,
        "d_ff": 1024,
        "num_layers": 6,
        "num_decoder_layers": 6,
        "num_heads": 8,
    },
    # P100で回しやすい範囲での大型化（推奨）
    "p100-medium": {
        "d_model": 512,
        "d_ff": 2048,
        "num_layers": 12,
        "num_decoder_layers": 12,
        "num_heads": 8,
    },
    # さらに大型（t5-base相当）。OOMしたら MAX_LENGTH を下げる
    "t5-base-p100": {
        "d_model": 768,
        "d_ff": 3072,
        "num_layers": 12,
        "num_decoder_layers": 12,
        "num_heads": 12,
    },
}

if Config.MODEL_PRESET not in MODEL_PRESETS:
    raise ValueError(f"Unknown MODEL_PRESET: {Config.MODEL_PRESET}. choices={list(MODEL_PRESETS)}")

preset = MODEL_PRESETS[Config.MODEL_PRESET]

config = T5Config(
    vocab_size=tokenizer.vocab_size,
    dropout_rate=0.1,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    decoder_start_token_id=tokenizer.pad_token_id,
    **preset,
)

model = T5ForConditionalGeneration(config)

# 大型モデルはメモリが厳しいのでチェックポイント化
model.gradient_checkpointing_enable()
model.config.use_cache = False

model.generation_config.max_length = Config.MAX_LENGTH


# Create Batched Dataset

In [ ]:
from datasets import Dataset

def preprocess(examples):
    prefix = "translate Akkadian to English: "

    src_norm = _preprocessor.preprocess_batch(examples[SRC_COL])
    tgt_norm = _postprocessor.postprocess_batch(examples[TGT_COL])

    inputs = [prefix + ex for ex in src_norm]
    targets = [ex for ex in tgt_norm]

    model_inputs = tokenizer(
        inputs,
        max_length=Config.MAX_LENGTH,
        truncation=True,
        text_target=targets
    )
    return model_inputs
    
dataset = Dataset.from_pandas(train_df)
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)

train_dataset = split_dataset["train"].map(preprocess, batched=True)
val_dataset = split_dataset["test"].map(preprocess, batched=True)


# Training

In [ ]:
from sacrebleu.metrics import CHRF
import numpy as np

# update the metrics as competition metrics collapse during training phase
chrf = CHRF(word_order=2)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_preds = [postprocess_translation(p) for p in decoded_preds]

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_labels = [postprocess_translation(l) for l in decoded_labels]

    # sentence-level chrF (macro)
    sent_scores = [
        chrf.sentence_score(p.strip(), [l.strip()]).score
        for p, l in zip(decoded_preds, decoded_labels)
    ]

    return {
        "sentence_chrf": float(np.mean(sent_scores))
    }


In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback, GenerationConfig

# GenerationConfig(
#     max_length=1024,
#     num_beams=4,
#     no_repeat_ngram_size=4,
#     repetition_penalty=1.3,
#     length_penalty=1.1,
#     early_stopping=True,
# )

EPOCH = 3 if Config.DRY_RUN else Config.EPOCHS

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=Config.LEARNING_RATE,      # higher LR since training from scratch
    per_device_train_batch_size=Config.BATCH_SIZE,
    per_device_eval_batch_size=Config.BATCH_SIZE,
    gradient_accumulation_steps=Config.GRAD_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    optim="adafactor",
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=EPOCH,
    fp16=True,
    logging_steps=100,
    report_to="none",
    # ===== EARLY STOPPING CONFIG =====
    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,
    greater_is_better=False,
    # predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    # early stopping
    # compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=Config.PATIENCE)],

)

trainer.train()


# Save

In [ ]:
trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)
print(f"Model saved to {Config.OUTPUT_DIR}")

In [ ]:
# predictions = trainer.predict(val_dataset)
# labels = predictions.label_ids
# labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
# decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

# decoded_labels = [[label.strip()] for label in decoded_labels]
# pred_ids = predictions.predictions

# if isinstance(pred_ids, tuple):
#     pred_ids = pred_ids[0]

# decoded_preds = tokenizer.batch_decode(
#     pred_ids,
#     skip_special_tokens=True,
#     clean_up_tokenization_spaces=True,
# )
# decoded_preds = [pred.strip() for pred in decoded_preds]

# Dry run Inference

In [ ]:
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

MAX_LENGTH = Config.MAX_LENGTH
DEVICE = "cuda"

tokenizer_loaded = AutoTokenizer.from_pretrained(Config.OUTPUT_DIR)
model_loaded = AutoModelForSeq2SeqLM.from_pretrained(Config.OUTPUT_DIR).to("cuda")
model_loaded.generation_config.max_length = Config.MAX_LENGTH
model_loaded.eval()


class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts = df['transliteration'].astype(str).tolist()
        self.texts = _preprocessor.preprocess_batch(self.texts)
        self.texts = ["translate Akkadian to English: " + i for i in self.texts]
        self.translation = df['translation'].astype(str).tolist()
        self.translation = _postprocessor.postprocess_batch(self.translation)
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        translation = self.translation[idx]
        inputs = self.tokenizer(
            text, 
            max_length=MAX_LENGTH, 
            padding="max_length", 
            truncation=True, 
            return_tensors="pt"
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "translation": translation
        }

test_df = train_df.sample(5)
test_dataset = InferenceDataset(test_df, tokenizer_loaded)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


print("Starting Inference...")
all_predictions = []
actuals = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        actuals.extend(batch['translation'])
  
        outputs = model_loaded.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            do_sample=False,
            # num_beams=4,
            # early_stopping=True,
            # top_k=30, 
            # top_p=0.95
        )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_predictions.extend([postprocess_translation(d).strip() for d in decoded])

pred = pd.DataFrame(
    {
        "predictions": all_predictions,
        "actuals": actuals
    }
    
)
pred
